# Processing Template
___
Jonathan Zhang

Here, we outline how to use this notebook to process stitched and background-subtracted images from HT-BAM experiments. Before you start, carefully read the guidelines below.

## General Notes

### Metadata framework
There exist two sources of metadata to bridge the gap between raw pixels and experimental variables:
- `imaging.csv`: This file logs hardware state and raster metadata for all images taken in an experiment. All information needed to stitch and background subtract images is found here. You may manually overwrite this file to alter how images are processed, but always be sure to create a backup copy beforehand.
    - For the sake of continuity, the stitching and background subtraction modules carry over this data into `stitched_images.csv` and `bgsub_images.csv`.
- `series_index.json`: Metadata specific to the experimental conditions (e.g., titration concentrations) is saved within individual series directories. This ensures that raw data remain linked to their biochemical context. This file is largely irrelevant during stitching, but become important at the processing stage.

### Before you begin
- Manually delete any images you do not want processed beforehand.
- Previous versions of the stitching code required manually directory reorginization. In this version, reorganizing the directory structure is entirely unnecessary and will likely cause the code to fail.
    - Do not rename, move, or create new files within directories of raw images or outputs from the stitching and background subtraction modules (i.e. `raw_images`, `stitched_images`, and `bgsub_images`).
- If you are running processing on a laptop, probably a good idea to close all uneeded applications (code is both memory and CPU intensive).


In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from htbam_analysis.stitching import ImageStitcher, BackgroundSubtractor
from htbam_analysis.processing import Processor
from htbam_analysis.processing.experiment import Experiment, Device, DataHandler

from pathlib import Path
import pandas as pd
import numpy as np

# to mute annoying numpy warnings
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Image Stitching
The stitching process is designed to be automated and data-driven:
- `stitch_images`: Once the root path is provided, this function acts as the primary orchestrator. It uses the raster parameters defined in `imaging.csv` to find and stitch all raw data associated with the experiment. There are three arguments:
    1. `root`: path to the folder containing experiment data.
    2. `rotation`: global rotation parameter. This can drift over time and may need to be adjusted. If stitched images look jagged, this is the first thing that should be tuned.
    3. `acqui_ori`: specifies corner of the device where imaging starts. In our experiments, this is always from the top-right (i.e. `(True, False)`) 
- `stitch_single_raster`: This is the lower-level function called by the main stitcher. If your file-naming convention or folder structure changes substantially from convention, you can bypass the automated discovery logic implemented in `stitch_images` and use this function directly by providing your own path-finding logic.

NOTE: by default, stitched images are not flat-field corrected. If one desires to implement correction for all or a subset of images, one can "hack" the ImageStitcher by manually updating the `apply_ff_correction` column in the `raster_data` attribute of the `ImageStitcher`. For example:

```python
# configure ff correction for just button quant images
bq_mask = stitcher.raster_data['image_path'].apply(str).str.contains('2026-03-09_15-16-09_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant')
stitcher.raster_data.loc[bq_mask, 'apply_ff_correction'] = True
```

In [126]:
# init stitcher
root = Path('/Volumes/JSZ2/20260309')
stitcher = ImageStitcher(root)

# # configure ff correction for just button quant images
# bq_mask = stitcher.raster_data['image_path'].apply(str).str.contains('aura_cyan_2_5_sensitivity_2x2_1')
# stitcher.raster_data.loc[bq_mask, 'apply_ff_correction'] = True

In [ ]:
# OPTIONAL: test different rotation parameters to find the ideal one 
# stitcher.test_stitching_rotations(
#     raster_path = '/Volumes/JSZ2/20260309/raw_images/2026-03-09_17-07-43_d2_aura_UV340_5_500_dynamic_range_2x2_4_pre_kinetics_brightfield',
#     rotations = [-2.2, -2.15, -2.1, -2.05, -2, -1.95, -1.9],
#     acqui_origin = (True, False),
#     outdir = root / 'stitch_test'
# )

In [ ]:
# stitch the images
stitcher.stitch_images(rotation=-2.05, acqui_origin=(True, False))

## Background Subtraction

Like stitching, background subtraction logic is split into two layers to balance automation with developer flexibility:
- `background_subtract_images`: The main function for background subtraction. It automatically groups images based on specified metadata headers (like channel or exposure_time) and applies the subtraction across the entire experiment. It uses `stitched_images.csv` as the source for metadata.
- `background_subtract`: The lower-level function, called within the above, which performs the background subtraction operation on a single image or array. If your file structure changes substantially or you need to implement custom subtraction logic, you can call this function directly and bypass the automated grouping.

### Note on Image Grouping
Users can specify metadata headers in `stitched_images.csv` to use for image grouping (i.e. exposure time, lightsource, etc.). If complex groupings are required, one can manually write new columns to `stitched_images.csv` to define custom logic (but always save a backup copy beforehand).

In [ ]:
# paths to all background images
# use full, absolute paths
background_images = [
    '/Volumes/JSZ2/20260309/stitched_images/2026-03-09_11-36-50_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant_background.tif',
    '/Volumes/JSZ2/20260309/stitched_images/2026-03-09_15-19-34_d2_aura_green_3_50_sensitivity_2x2_1_mscarlet_background.tif',
    '/Volumes/JSZ2/20260309/stitched_images/2026-03-09_17-23-13_d2_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif',
]
settings_to_match = ['temp', 'hum','setup', 'dname', 'lightsource', 'channel', 'exposure', 'camera_mode', 'binning', 'nosepiece', 'apply_ff_correction']
subtractor = BackgroundSubtractor(root)
subtractor.subtract(
    background_images=background_images,
    settings_to_match=settings_to_match,
)

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d2 | lightsource: aura_UV340 | channel: 5 | exposure: 500 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4 | apply_ff_correction: False
Using /Volumes/JSZ2/20260309/stitched_images/2026-03-09_17-23-13_d2_aura_UV340_5_500_dynamic_range_2x2_4_nadh_background.tif as background image


Running background subtraction.: 100%|██████████| 259/259 [00:00<00:00, 5284.50it/s]


259 / 259 images were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d2 | lightsource: aura_cyan | channel: 2 | exposure: 5 | camera_mode: sensitivity | binning: 2x2 | nosepiece: 1 | apply_ff_correction: True
Using /Volumes/JSZ2/20260309/stitched_images/2026-03-09_11-36-50_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant_background.tif as background image


Running background subtraction.: 100%|██████████| 18/18 [00:00<00:00, 3171.23it/s]


18 / 18 images were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d2 | lightsource: aura_green | channel: 3 | exposure: 50 | camera_mode: sensitivity | binning: 2x2 | nosepiece: 1 | apply_ff_correction: False
Using /Volumes/JSZ2/20260309/stitched_images/2026-03-09_15-19-34_d2_aura_green_3_50_sensitivity_2x2_1_mscarlet_background.tif as background image


Running background subtraction.: 100%|██████████| 7/7 [00:00<00:00, 2477.02it/s]

7 / 7 images were successfully background subtracted



## Processing

### Initialization

To begin, initialize an `Experiment` object by providing the root directory of your data. You must then create `Device` objects, representing the devices used within an experiment, and then register them. 

When creating device objects, one can specify pinlists using the `set_pinlist` method. There are two kwargs that are helpful here:
    1. `block_descriptions`- a dictionary specifying which variants are present in each block (for flow-on experiments only)
    2. `pinlist_path`- path to a pinlist file output by `Array-Print` (for expression experiments only) 

For image and metadata management, we use the `DataHandler` class. This class is responsible for fetching images and associated metadata to be processed.

### Processing logic

The execution logic is as follows:
1. Retrieve image data and metadata to be processed via methods implemented in the `DataHandler`.
    - This is done by providing image identifier(s) to the `get_images()` method within the `DataHandler` class.
2. Initialize a processor object.
    - In addition to your experiment object and image data from (1), you will need to designate the type of feature to process (i.e. 'button', 'chamber', or 'all'). 
3. Optionally, set reference images for each device.
    - You'll need to designate the coordinates of the center of all four corner buttons (see below).
4. Process images using the core `process()` method.
    - If you don't have a reference image, pass a single set of coordinates OR a list of N coordinates (N being the number of images to be processed).

Poor feature finding results arise from two problems (almost all the time):
1. Jagged stitches. This can be fixed by going back to stitching and optimizing the rotation parameter.
2. Bad corner picks. Pick new corners.

### A note on corner picking

We have a command line tool for picking corners. Within your terminal, run the following command (be sure to activate the correct conda environment beforehand):

```bash
pick-corners /path/to/stitched/and/subtracted/image.tif
```

In [129]:
root = Path('/Volumes/JSZ2/deez')

# initialize experiment object
experiment = Experiment(root)

# initialize device(s)
# NOTE: dname NEEDS to exactly match that used within the experiment
d1 = Device(setup='s1', dname='d1', dims=(32, 56))
d1.set_pinlist(block_descriptions={1: 'BLANK', 2: 'BLANK', 3: 'HRAS', 4: 'HRAS'})

d2 = Device(setup='s1', dname='d2', dims=(32, 56))
d2.set_pinlist(block_descriptions={1: 'BLANK', 2: 'BLANK', 3: 'KRAS', 4: 'KRAS'})

# add devices to experiment
experiment.add_device(d1)
experiment.add_device(d2)

# init data handler object
# will print out identifiers that can be passed into the get_images method
data_handler = DataHandler(root)

Loaded the following image sets:
2026-03-05_12-26-07_standard_curve_NADH
2026-03-05_13-17-36_kinetic_series_cycling

Loaded the following images:
2026-03-05_12-10-19_d1_aura_UV340_5_500_dynamic_range_2x2_4_pre_standard_brightfield
2026-03-05_12-10-50_d2_aura_UV340_5_500_dynamic_range_2x2_4_pre_standard_brightfield


## Button Quant Processing

The `identifiers` argument can either be a list of identifiers (printed above by the `DataHandler` class), or a single identifier. All images associated with the identifier will be loaded.

In [128]:
# copy/paste identifiers printed by DataHandler above
button_quant_identifiers = [
    '2026-03-09_15-16-09_d2_aura_cyan_2_5_sensitivity_2x2_1_button_quant'
]

# corners for each image (use pick-corners app here)
corners = [
    ((491, 413), (6681, 425), (484, 6702), (6672, 6703))
]

# get image data
button_quant_image_data = data_handler.get_images(identifiers=button_quant_identifiers)
button_quant_image_data

# init the processor
button_quant_processor = Processor(
    experiment= experiment,
    image_data = button_quant_image_data,
    features = 'button'
)

# process the data
button_quant_df = button_quant_processor.process(
    corners = corners,
    summary_image_dir = root / 'summary_images'
)

# save the data to disc
button_quant_df.to_csv(root / 'data_analysis' / 'button_quant.csv')

Finding Buttons: 100%|██████████| 1792/1792 [03:15<00:00,  9.16it/s]
Processing images: 1it [03:16, 196.81s/it]


## Standard Curve Processing

In [136]:
standard_images = data_handler.get_images(identifiers='2026-03-05_12-26-07_standard_curve_NADH')

standard_processor = Processor(experiment, image_data=standard_images, features='chamber')
corners = ((458, 438)), ((6645, 437)), ((460, 6714)), ((6652, 6718))

# set references (for both devices)
standard_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_12-26-07_standard_curve_NADH/2026-03-05_13-11-23_d1_aura_UV340_5_1000_dynamic_range_2x2_4_standard_curve_NADH_7.tif', 
    corners=corners, 
    summary_image_path=root / 'summary_images' / 'standard_curve_chamber_summary_d1.tif'
    )
standard_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_12-26-07_standard_curve_NADH/2026-03-05_13-12-31_d2_aura_UV340_5_1000_dynamic_range_2x2_4_standard_curve_NADH_7.tif', 
    corners=corners, 
    summary_image_path=root / 'summary_images' / 'standard_curve_chamber_summary_d2.tif'
    )

In [140]:
standard_df = standard_processor.process(use_reference=True)

Processing images: 100%|██████████| 32/32 [01:31<00:00,  2.86s/it]


In [143]:
# add your desired output path here
standard_df.to_csv(root / 'standard_data.csv.bz2', compression='bz2')

## Kinetics Processing

In [ ]:
kinetics_images = data_handler.get_images(identifier='2026-03-05_13-17-36_kinetic_series_cycling')

kinetics_processor = Processor(experiment, image_data=kinetics_images, features='chamber')
corners = ((457, 437)), ((6644, 434)), ((464, 6716)), ((6649, 6714))

# set references (for both devices)
kinetics_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_13-17-36_kinetic_series_cycling/2026-03-05_13-17-36_timecourse_0/2026-03-05_13-17-49_d1_aura_UV340_5_1000_dynamic_range_2x2_4_kinetic_series_cycling_timecourse_0_0.tif', 
    corners=corners, 
    summary_image_path=root / 'kinetics_chamber_summary_d1.tif'
    )
kinetics_processor.set_reference(
    image='/Volumes/JSZ2/deez/bgsub_images/2026-03-05_13-17-36_kinetic_series_cycling/2026-03-05_13-17-36_timecourse_0/2026-03-05_13-18-57_d2_aura_UV340_5_1000_dynamic_range_2x2_4_kinetic_series_cycling_timecourse_0_0.tif', 
    corners=corners, 
    summary_image_path=root / 'kinetics_chamber_summary_d2.tif'
    )

In [171]:
kinetics_df = kinetics_processor.process(use_reference=True, summary_image_dir=root / 'kinetics_chamber_summaries')

Processing images: 100%|██████████| 24/24 [01:10<00:00,  2.93s/it]


In [172]:
# add your desired output path here
kinetics_df.to_csv(root / 'kinetics_data.csv.bz2', compression='bz2')